# RSNA Knee — train teacher_pseudo (student)

| Experiment | Notebook | What this version does |
|---|---|---|
| **EXP-003** | `hlly34/rsna-knee-train-student` | student: same model, targets = 50/50 report labels + teacher OOF (Synovitis 70%, PF/Lateral OA 60%) |

Trains every fold of one config. Folds come from the committed `configs/folds_5.csv` (same split in every round; the 58 gold studies are never trained on). Each fold saves a top-3-epoch weight-averaged model, its out-of-fold predictions and gold predictions; the fold predictions are then merged into a CV summary. With 2 GPUs two folds train at once.

**Pseudo-labels:** targets are `(1-w)·report + w·teacher_OOF` using the out-of-fold predictions of `hlly34/rsna-knee-train-teacher` (EXP-002). Each training study's teacher prediction comes from the fold model that did not see it.

Pipeline code lives in the attached dataset `rsna-knee-code` (repo `src/`, `configs/`, `scripts/`); this notebook only orchestrates it. Results are registered with `scripts/track.py` into `experiments/registry.csv` using the id above.

In [ ]:
EXP = {"id": "EXP-003", "notebook": "hlly34/rsna-knee-train-student", "what": "student: same model, targets = 50/50 report labels + teacher OOF (Synovitis 70%, PF/Lateral OA 60%)"}
print("EXPERIMENT", EXP["id"], "-", EXP["what"])
CFG, EPOCHS, FOLDS, EXTRA = 'teacher_pseudo', 15, [0, 1, 2, 3, 4], []
import glob, json, os, subprocess, sys, time

def walk_find(pred, root="/kaggle/input", max_depth=6):
    """Depth-limited search of the attached inputs that never descends into the DICOM trees."""
    root = root.rstrip("/")
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else sorted(d for d in dirnames if not d.endswith("_series"))
        hit = pred(dirpath, filenames)
        if hit:
            return hit
    return None

# Kaggle mounts inputs at /kaggle/input/{competitions,datasets,notebooks}/... - locate them instead of hardcoding
CODE = walk_find(lambda d, f: d if os.path.isfile(os.path.join(d, "src", "models", "inference.py")) else None)
COMP = walk_find(lambda d, f: d if "test.csv" in f and os.path.isdir(os.path.join(d, "test_series")) else None)
assert CODE and COMP, (CODE, COMP)
sys.path.insert(0, CODE)
print("code:", CODE, "| competition:", COMP)


## 1. Environment and inputs

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification", "lightning",
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=True)
import torch, timm, pydicom
print("torch", torch.__version__, "| timm", timm.__version__, "| pydicom", pydicom.__version__,
      "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "| CPUs:", os.cpu_count())
DATA = "/kaggle/tmp/data"            # scratch space (not saved); outputs go to /kaggle/working
os.makedirs(f"{DATA}/labels", exist_ok=True)
if not os.path.exists(f"{DATA}/raw"):
    os.symlink(COMP, f"{DATA}/raw")
for name in ["llm_labels_v4_blend.csv", "llm_labels_full.csv", "report_labels_v2.csv"]:
    src = walk_find(lambda d, f: os.path.join(d, name) if name in f else None)
    print(f"{name:28s} <- {src}")
    if src and not os.path.exists(f"{DATA}/labels/{name}"):
        os.symlink(src, f"{DATA}/labels/{name}")
ENV = dict(os.environ, DATA_DIR=DATA, PYTHONPATH=CODE)
os.chdir("/kaggle/working")

def run(*args, env=None, log=None):
    """Run a repo script/module as a subprocess (fresh CUDA context, clean logs)."""
    t = time.time()
    out = open(log, "w") if log else None
    subprocess.run([sys.executable, *args], env=env or ENV, check=True, stdout=out, stderr=subprocess.STDOUT if out else None)
    print(f"[{' '.join(map(str, args[:2]))}] {time.time() - t:.0f}s")

CACHE = walk_find(lambda d, f: d if os.path.basename(d) == "cache_256" and any(x.endswith(".npz") for x in f[:50]) else None)
print("cache:", CACHE, len(os.listdir(CACHE)), "studies")
os.makedirs("/kaggle/working/wheels", exist_ok=True)   # offline wheels for the inference notebook
subprocess.run([sys.executable, "-m", "pip", "download", "-q", "-d", "/kaggle/working/wheels",
                "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg"], check=False)
EXTRA = EXTRA + [f"paths.cache_dir={CACHE}"]

## 2. Teacher out-of-fold predictions → pseudo-labels

In [ ]:
CV_JSON = walk_find(lambda d, f: next((os.path.join(d, x) for x in f if x.endswith("_cv.json")), None))
TEACHER_OOF = CV_JSON.replace("_cv.json", ".csv")
print("teacher:", json.load(open(CV_JSON)))
print("teacher OOF:", TEACHER_OOF)
EXTRA = EXTRA + [f"labels.pseudo.oof_files=[{TEACHER_OOF}]"]
run(f"{CODE}/scripts/eval_labels.py", f"exp={CFG}", *EXTRA)   # last line = targets incl. pseudo-labels vs gold

## 3. Train all folds
Each fold is a separate process (logs in `/kaggle/working/logs/fold*.txt`); folds are distributed over the available GPUs.

In [ ]:
import torch
n_gpu = max(1, torch.cuda.device_count())
workers = max(2, os.cpu_count() // n_gpu)
queue, running = list(FOLDS), []
os.makedirs("logs", exist_ok=True)
t0 = time.time()
while queue or running:
    while queue and len(running) < n_gpu:
        fold = queue.pop(0)
        gpu = [g for g in range(n_gpu) if g not in [r[2] for r in running]][0]
        env = dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))
        cmd = [sys.executable, "-m", "src.models.train_pl", f"exp={CFG}", f"fold={fold}",
               f"training.num_epochs={EPOCHS}", f"resources.num_workers={workers}", *EXTRA]
        running.append((subprocess.Popen(cmd, env=env, stdout=open(f"logs/fold{fold}.txt", "w"),
                                         stderr=subprocess.STDOUT), fold, gpu))
        print(f"{(time.time() - t0) / 60:6.1f} min  start fold {fold} on GPU {gpu}")
    for r in list(running):
        if r[0].poll() is not None:
            running.remove(r)
            print(f"{(time.time() - t0) / 60:6.1f} min  fold {r[1]} exit {r[0].returncode}")
            print("".join(l for l in open(f"logs/fold{r[1]}.txt") if "macro AUC" in l or "Error" in l)[-2000:])
            assert r[0].returncode == 0, open(f"logs/fold{r[1]}.txt").read()[-5000:]
    time.sleep(20)

## 4. Cross-validation summary
* **oof_auc** — all folds' out-of-fold predictions pooled, vs report targets (noise ≈ 0.003).
* **gold_auc_ensemble** — the 5 fold models rank-averaged on the 58 gold studies, i.e. what the submission ensemble does (noise ≈ ±0.02).

In [ ]:
exp_name = json.load(open(sorted(glob.glob("models/*/fold*_metrics.json"))[0]))["exp_name"]
run(f"{CODE}/scripts/merge_oof.py", "--exp", exp_name, "--oof_dir", "models/oof",
    "--train", f"{COMP}/train.csv", "--targets", "models/oof/report_targets.csv")
cv = json.load(open(f"models/oof/{exp_name}_cv.json"))
import pandas as pd
print(pd.DataFrame({k: cv[k] for k in ("oof_per_label", "gold_per_label") if k in cv}).round(3).T.to_string())
EXP.update({k: v for k, v in cv.items() if "per_label" not in k}, minutes=round((time.time() - t0) / 60, 1))
json.dump(EXP, open("/kaggle/working/experiment.json", "w"), indent=1)
print(EXP)